# 🎵 TikTok Creator Data Analysis
**Goal:** Understand what drives TikTok engagement using a real creator dataset.  
**Methods:** EDA · Hashtag frequency · Trend analysis (pytrends) · Caption NLP · Niche heatmaps · Sentiment vs engagement

---

## 0. Setup & Imports

In [ ]:
# Install dependencies (run once)
# !pip install pandas numpy matplotlib seaborn wordcloud pytrends transformers scikit-learn

import os
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from collections import Counter
from datetime import datetime, timedelta

warnings.filterwarnings('ignore')

# TikTok-style dark theme
plt.rcParams.update({
    'figure.facecolor':  '#0f0f0f',
    'axes.facecolor':    '#1a1a1a',
    'axes.edgecolor':    '#333',
    'text.color':        'white',
    'axes.labelcolor':   'white',
    'xtick.color':       'white',
    'ytick.color':       'white',
    'grid.color':        '#2a2a2a',
    'font.family':       'DejaVu Sans',
})
TIKTOK_COLORS = ['#fe2c55', '#25f4ee', '#ffffff', '#fffc00', '#69c9d0', '#ff6b6b', '#4ecdc4']

print('Setup complete ✅')

## 1. Load Dataset

We use a **synthetic dataset** that mirrors real TikTok metrics distributions.  
To use your own data: replace the `generate_synthetic_dataset()` call with `pd.read_csv('your_file.csv')`.

**Dataset source options:**
- [Kaggle: TikTok Videos Dataset](https://www.kaggle.com/datasets/yakhyojon/tiktok)
- [Kaggle: TikTok User Analytics](https://www.kaggle.com/datasets/datasets?search=tiktok)
- Export your own via TikTok Creator Portal → Analytics

In [ ]:
import random

random.seed(42)
np.random.seed(42)

NICHES = [
    'Fitness', 'Beauty', 'Food', 'Comedy', 'Finance',
    'Travel', 'Dance', 'Study', 'Fashion', 'Mental Health'
]

HASHTAG_POOL = {
    'Fitness':       ['#fitness', '#gym', '#workout', '#gains', '#fitspo', '#grwm', '#bodytransformation'],
    'Beauty':        ['#makeup', '#grwm', '#skincare', '#beauty', '#tutorial', '#glam', '#fyp'],
    'Food':          ['#food', '#recipe', '#cooking', '#foodie', '#whatieatinaday', '#mealprep', '#fyp'],
    'Comedy':        ['#funny', '#comedy', '#pov', '#skit', '#relatable', '#fyp', '#foryou'],
    'Finance':       ['#finance', '#money', '#investing', '#savingmoney', '#budgeting', '#richlife', '#fyp'],
    'Travel':        ['#travel', '#wanderlust', '#explore', '#aesthetic', '#dayinmylife', '#vlog', '#fyp'],
    'Dance':         ['#dance', '#viral', '#trending', '#choreo', '#dancetrend', '#fyp', '#foryoupage'],
    'Study':         ['#studywithme', '#student', '#revision', '#university', '#notes', '#academic', '#fyp'],
    'Fashion':       ['#fashion', '#ootd', '#style', '#aesthetic', '#grwm', '#outfit', '#fyp'],
    'Mental Health': ['#mentalhealth', '#selfcare', '#wellness', '#mindset', '#healing', '#anxiety', '#fyp'],
}

CAPTIONS = {
    'Fitness':  ['POV: Day 30 of my body transformation 💪', 'I tried working out for 30 days and this happened',
                 'The gym glow-up is real 🔥', 'No one told me it would be this hard...'],
    'Beauty':   ['Get ready with me for my first date 💄', 'This £3 dupe is better than the original',
                 'POV: Skincare that actually works', 'I tested viral TikTok makeup hacks'],
    'Food':     ['I cooked a £2 meal and it slapped 🍳', 'Viral pasta recipe you NEED to try',
                 'What I eat in a day as a broke student', 'Gordon Ramsay approved this (probably)'],
    'Comedy':   ['When your mum tries to understand TikTok 😭', 'POV: You are me at 3am',
                 'Things that live in my head rent free', 'UK vs US be like...'],
    'Finance':  ['I saved £5k in 6 months on minimum wage', 'Things broke people say',
                 'Investing £50 a month for 10 years = ?', 'Stop doing this with your money'],
    'Travel':   ['I spent £500 travelling Europe for 2 weeks', 'Hidden gems in London no one talks about',
                 'Day in my life in Tokyo 🇯🇵', 'Budget travel hacks that actually work'],
    'Dance':    ['Obsessed with this new trending sound 🎵', 'Teaching my mum this dance 😭',
                 'When the beat drops perfectly 🔥', 'Choreo to the song everyone is using'],
    'Study':    ['Study with me for my finals 📚', '6 hour productive day in my life at uni',
                 'Note-taking method that changed everything', 'Failing my degree (not clickbait)'],
    'Fashion':  ['Get ready with me for a night out ✨', 'Thrifted this entire outfit for £12',
                 'GRWM: first day back at uni', 'Outfit ideas for autumn 2026'],
    'Mental Health': ['Things I wish someone told me about anxiety', 'Healing era check-in 🧠',
                      'You are not lazy, you are burnt out', 'POV: Bad mental health day'],
}

def generate_synthetic_dataset(n=500):
    records = []
    for i in range(n):
        niche     = random.choice(NICHES)
        duration  = random.choice([15, 30, 60, 90, 180])
        followers = int(np.random.lognormal(9, 1.2))
        followers = max(500, min(followers, 5_000_000))

        # Simulate virality: ~5% of posts go viral
        viral     = random.random() < 0.05
        views_base = followers * random.uniform(0.3, 3.0)
        if viral:
            views_base *= random.uniform(10, 80)
        views     = max(100, int(views_base * random.gauss(1, 0.3)))

        like_rate    = random.uniform(0.04, 0.18)
        comment_rate = random.uniform(0.003, 0.025)
        share_rate   = random.uniform(0.005, 0.04)
        save_rate    = random.uniform(0.008, 0.06)

        likes    = int(views * like_rate)
        comments = int(views * comment_rate)
        shares   = int(views * share_rate)
        saves    = int(views * save_rate)

        n_hashtags  = random.randint(3, 20)
        pool        = HASHTAG_POOL[niche]
        common_hash = ['#fyp', '#foryou', '#viral', '#trending']
        hashtags    = random.sample(pool, min(len(pool), n_hashtags // 2))
        hashtags   += random.sample(common_hash, min(len(common_hash), n_hashtags - len(hashtags)))
        hashtag_str = ' '.join(hashtags[:n_hashtags])

        caption_base = random.choice(CAPTIONS[niche])
        caption      = caption_base + ' ' + hashtag_str

        post_date = datetime(2025, 1, 1) + timedelta(days=random.randint(0, 530))
        post_hour = random.choices(
            range(24),
            weights=[1,1,1,1,1,1,2,4,5,4,3,4,6,5,4,5,8,10,12,10,8,6,4,2],
            k=1
        )[0]

        records.append({
            'post_id':       f'post_{i:05d}',
            'niche':         niche,
            'caption':       caption,
            'caption_base':  caption_base,
            'hashtags':      hashtag_str,
            'n_hashtags':    n_hashtags,
            'caption_len':   len(caption_base),
            'duration_sec':  duration,
            'post_date':     post_date,
            'post_hour':     post_hour,
            'post_day':      post_date.strftime('%A'),
            'followers':     followers,
            'views':         views,
            'likes':         likes,
            'comments':      comments,
            'shares':        shares,
            'saves':         saves,
            'is_viral':      viral,
        })

    df = pd.DataFrame(records)
    df['engagement_rate'] = (df['likes'] + df['comments'] + df['shares'] + df['saves']) / df['views'] * 100
    df['views_per_follower'] = df['views'] / df['followers']
    return df

df = generate_synthetic_dataset(500)
print(f'Dataset shape: {df.shape}')
print(f'\nNiche distribution:')
print(df['niche'].value_counts().to_string())
df.head()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Summary statistics
stats = df[['views', 'likes', 'comments', 'shares', 'saves', 'engagement_rate', 'followers']].describe()
print('=== Summary Statistics ===')
print(stats.round(1).to_string())

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle('📊 TikTok Metrics Distribution', fontsize=16, color='white', y=1.02)

metrics = ['views', 'likes', 'comments', 'shares', 'saves', 'engagement_rate']
titles  = ['Views', 'Likes', 'Comments', 'Shares', 'Saves', 'Engagement Rate (%)']
colors  = TIKTOK_COLORS

for ax, metric, title, color in zip(axes.flat, metrics, titles, colors):
    data = np.log1p(df[metric]) if metric != 'engagement_rate' else df[metric]
    ax.hist(data, bins=40, color=color, alpha=0.85, edgecolor='none')
    xlabel = f'log({metric})' if metric != 'engagement_rate' else metric
    ax.set_xlabel(xlabel, color='white', fontsize=10)
    ax.set_ylabel('Count', color='white', fontsize=10)
    ax.set_title(title, color='white', fontsize=11)
    median_val = data.median()
    ax.axvline(median_val, color='white', linestyle='--', linewidth=1, alpha=0.7, label=f'Median: {median_val:.1f}')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('tiktok_eda_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: tiktok_eda_distributions.png')

In [ ]:
# Views by niche — boxplot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

niche_order = df.groupby('niche')['views'].median().sort_values(ascending=False).index

# Box plot
data_by_niche = [np.log1p(df[df['niche'] == n]['views']) for n in niche_order]
bp = ax1.boxplot(data_by_niche, labels=niche_order, patch_artist=True,
                 medianprops={'color': '#fe2c55', 'linewidth': 2})
for i, patch in enumerate(bp['boxes']):
    patch.set_facecolor(TIKTOK_COLORS[i % len(TIKTOK_COLORS)])
    patch.set_alpha(0.7)
ax1.set_title('Views Distribution by Niche (log scale)', color='white')
ax1.set_xlabel('Niche', color='white')
ax1.set_ylabel('log(Views)', color='white')
plt.setp(ax1.get_xticklabels(), rotation=45, ha='right')

# Mean engagement rate by niche
er_by_niche = df.groupby('niche')['engagement_rate'].mean().sort_values(ascending=True)
bars = ax2.barh(er_by_niche.index, er_by_niche.values,
                color=[TIKTOK_COLORS[i % len(TIKTOK_COLORS)] for i in range(len(er_by_niche))])
ax2.set_title('Average Engagement Rate by Niche (%)', color='white')
ax2.set_xlabel('Engagement Rate (%)', color='white')
for bar, val in zip(bars, er_by_niche.values):
    ax2.text(val + 0.05, bar.get_y() + bar.get_height()/2,
             f'{val:.1f}%', va='center', color='white', fontsize=9)

plt.tight_layout()
plt.savefig('tiktok_niche_performance.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nTop 3 niches by engagement rate:')
print(er_by_niche.sort_values(ascending=False).head(3).to_string())

## 3. Hashtag Frequency Analysis

In [ ]:
# Extract all hashtags
all_hashtags = []
for row in df['hashtags']:
    tags = re.findall(r'#\w+', str(row).lower())
    all_hashtags.extend(tags)

hashtag_counts = Counter(all_hashtags)
top_hashtags   = pd.DataFrame(hashtag_counts.most_common(30), columns=['hashtag', 'count'])

print(f'Total hashtag instances: {len(all_hashtags)}')
print(f'Unique hashtags: {len(hashtag_counts)}')
print(f'\nTop 20 hashtags:')
print(top_hashtags.head(20).to_string(index=False))

In [ ]:
# Word cloud
try:
    from wordcloud import WordCloud

    wc = WordCloud(
        width=900, height=450, background_color='#0f0f0f',
        colormap='RdYlCy', max_words=80, prefer_horizontal=0.8,
        min_font_size=10,
    ).generate_from_frequencies(hashtag_counts)

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title('#️⃣  TikTok Hashtag Word Cloud', color='white', fontsize=14, pad=15)
    plt.tight_layout()
    plt.savefig('tiktok_hashtag_wordcloud.png', dpi=150, bbox_inches='tight',
                facecolor='#0f0f0f')
    plt.show()
    print('Saved: tiktok_hashtag_wordcloud.png')
except ImportError:
    print('wordcloud not installed — run: pip install wordcloud')
    print('Showing bar chart instead:')

# Bar chart of top 20
fig, ax = plt.subplots(figsize=(12, 7))
top20 = top_hashtags.head(20)
bars = ax.barh(top20['hashtag'][::-1], top20['count'][::-1],
               color=['#fe2c55' if '#fyp' in t or '#viral' in t or '#trending' in t
                      else '#25f4ee' for t in top20['hashtag'][::-1]])
ax.set_title('Top 20 Most Used Hashtags', color='white', fontsize=13)
ax.set_xlabel('Frequency', color='white')
for bar, val in zip(bars, top20['count'][::-1]):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2,
            str(val), va='center', color='white', fontsize=9)
plt.tight_layout()
plt.savefig('tiktok_hashtag_bar.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Does hashtag count affect engagement?
df['hashtag_bucket'] = pd.cut(df['n_hashtags'],
                               bins=[0, 5, 10, 15, 20, 50],
                               labels=['1-5', '6-10', '11-15', '16-20', '21+'])

hash_er = df.groupby('hashtag_bucket', observed=True)['engagement_rate'].agg(['mean', 'median', 'count'])
hash_er.columns = ['Mean ER%', 'Median ER%', 'Posts']

fig, ax = plt.subplots(figsize=(9, 5))
x = range(len(hash_er))
ax.bar([i - 0.2 for i in x], hash_er['Mean ER%'],   0.35, label='Mean ER',   color='#fe2c55', alpha=0.85)
ax.bar([i + 0.2 for i in x], hash_er['Median ER%'], 0.35, label='Median ER', color='#25f4ee', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(hash_er.index)
ax.set_xlabel('Number of Hashtags', color='white')
ax.set_ylabel('Engagement Rate (%)', color='white')
ax.set_title('Hashtag Count vs Engagement Rate', color='white', fontsize=13)
ax.legend()
plt.tight_layout()
plt.savefig('tiktok_hashtag_count_vs_er.png', dpi=150, bbox_inches='tight')
plt.show()
print(hash_er.round(2).to_string())

## 4. Google Trends Analysis (via pytrends)

In [ ]:
try:
    from pytrends.request import TrendReq

    pt = TrendReq(hl='en-GB', tz=0)

    keywords = ['GRWM', 'day in my life', 'get ready with me', 'vlog', 'aesthetic']
    pt.build_payload(keywords, timeframe='today 3-m', geo='GB')
    trends_df = pt.interest_over_time()

    if 'isPartial' in trends_df.columns:
        trends_df = trends_df.drop(columns=['isPartial'])

    # Regional interest
    pt.build_payload(['TikTok', 'TikTok creator'], timeframe='today 12-m', geo='GB')
    regional = pt.interest_by_region(resolution='REGION', inc_low_vol=True)
    regional = regional.sort_values('TikTok', ascending=False).head(10)

    LIVE_TRENDS = True
    print('✅ pytrends loaded — live Google Trends data')

except ImportError:
    print('pytrends not installed — using simulated trend data')
    print('Install: pip install pytrends')
    LIVE_TRENDS = False

except Exception as e:
    print(f'pytrends error (rate limit or network): {e}')
    print('Using simulated data instead')
    LIVE_TRENDS = False

if not LIVE_TRENDS:
    # Simulate realistic trend data
    dates = pd.date_range('2025-01-01', periods=90, freq='D')
    keywords = ['GRWM', 'day in my life', 'get ready with me', 'vlog', 'aesthetic']
    np.random.seed(42)
    trends_df = pd.DataFrame(
        {kw: np.clip(50 + np.cumsum(np.random.randn(90) * 3), 10, 100) for kw in keywords},
        index=dates
    )

In [ ]:
# Trend line chart
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 9))

for i, kw in enumerate(trends_df.columns):
    ax1.plot(trends_df.index, trends_df[kw], color=TIKTOK_COLORS[i],
             linewidth=2, label=kw, marker='o', markersize=2)
ax1.set_title('📈 TikTok Creator Keywords: Google Search Interest', color='white', fontsize=13)
ax1.set_ylabel('Search Interest (0–100)', color='white')
ax1.legend(loc='upper left', fontsize=9)
ax1.fill_between(trends_df.index, trends_df[trends_df.columns[0]], alpha=0.1, color=TIKTOK_COLORS[0])

# Average interest bar
avg = trends_df.mean().sort_values(ascending=True)
bars = ax2.barh(avg.index, avg.values,
                color=[TIKTOK_COLORS[i % len(TIKTOK_COLORS)] for i in range(len(avg))])
ax2.set_title('Average Search Interest (Past 90 Days)', color='white', fontsize=13)
ax2.set_xlabel('Average Interest', color='white')
for bar, val in zip(bars, avg.values):
    ax2.text(val + 0.3, bar.get_y() + bar.get_height()/2,
             f'{val:.1f}', va='center', color='white', fontsize=10)

plt.tight_layout(pad=2)
plt.savefig('tiktok_google_trends.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Caption Length vs Engagement

In [ ]:
df['caption_len_bucket'] = pd.cut(
    df['caption_len'],
    bins=[0, 30, 60, 100, 150, 300],
    labels=['<30 chars', '30–60', '60–100', '100–150', '150+']
)

cap_er = df.groupby('caption_len_bucket', observed=True).agg(
    mean_er   =('engagement_rate', 'mean'),
    median_er =('engagement_rate', 'median'),
    mean_views=('views', 'mean'),
    count     =('post_id', 'count')
).round(2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
x = range(len(cap_er))
ax.bar(x, cap_er['mean_er'], color='#fe2c55', alpha=0.85, label='Mean ER%')
ax.plot(x, cap_er['median_er'], 'o--', color='#25f4ee', linewidth=2, markersize=7, label='Median ER%')
ax.set_xticks(x)
ax.set_xticklabels(cap_er.index, rotation=15)
ax.set_title('Caption Length vs Engagement Rate', color='white', fontsize=12)
ax.set_ylabel('Engagement Rate (%)', color='white')
ax.legend()

ax = axes[1]
scatter_colors = [TIKTOK_COLORS[i % len(TIKTOK_COLORS)] for i in range(len(df))]
ax.scatter(df['caption_len'], df['engagement_rate'], alpha=0.3, s=15, c='#fe2c55')
# Trend line
z = np.polyfit(df['caption_len'], df['engagement_rate'], 1)
p = np.poly1d(z)
xs = np.linspace(df['caption_len'].min(), df['caption_len'].max(), 100)
ax.plot(xs, p(xs), color='#25f4ee', linewidth=2, label=f'Trend (slope={z[0]:.4f})')
ax.set_xlabel('Caption Length (chars)', color='white')
ax.set_ylabel('Engagement Rate (%)', color='white')
ax.set_title('Caption Length vs ER — Scatter + Trend', color='white', fontsize=12)
ax.legend()

plt.tight_layout()
plt.savefig('tiktok_caption_length_er.png', dpi=150, bbox_inches='tight')
plt.show()

print('Caption length vs engagement breakdown:')
print(cap_er.to_string())

## 6. Posting Time Analysis

In [ ]:
# Hourly and day-of-week engagement heatmap
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
df['post_day'] = pd.Categorical(df['post_day'], categories=day_order, ordered=True)

heat_data = df.pivot_table(
    index='post_day',
    columns='post_hour',
    values='engagement_rate',
    aggfunc='mean'
)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 10))

# Heatmap
sns.heatmap(heat_data, ax=ax1, cmap='RdYlGn', annot=False,
            linewidths=0.3, linecolor='#111',
            cbar_kws={'label': 'Avg Engagement Rate (%)'})
ax1.set_title('🕐 Best Time to Post — Engagement Rate Heatmap', color='white', fontsize=13)
ax1.set_xlabel('Hour of Day (0 = midnight)', color='white')
ax1.set_ylabel('Day of Week', color='white')

# Hourly bar chart
hourly_er = df.groupby('post_hour')['engagement_rate'].mean()
peak_hour = hourly_er.idxmax()
bar_colors = ['#fe2c55' if h == peak_hour else '#25f4ee' for h in hourly_er.index]
ax2.bar(hourly_er.index, hourly_er.values, color=bar_colors, alpha=0.85)
ax2.set_xlabel('Hour of Day', color='white')
ax2.set_ylabel('Average Engagement Rate (%)', color='white')
ax2.set_title(f'Average Engagement by Posting Hour  (Peak: {peak_hour}:00)', color='white', fontsize=13)
ax2.set_xticks(range(0, 24, 2))
ax2.set_xticklabels([f'{h}:00' for h in range(0, 24, 2)], rotation=30)

plt.tight_layout()
plt.savefig('tiktok_posting_time_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Best posting hour: {peak_hour}:00')
best_day = df.groupby('post_day', observed=True)['engagement_rate'].mean().idxmax()
print(f'Best posting day: {best_day}')

## 7. Niche Performance Heatmap

In [ ]:
niche_metrics = df.groupby('niche').agg(
    avg_views         =('views', 'mean'),
    avg_likes         =('likes', 'mean'),
    avg_comments      =('comments', 'mean'),
    avg_shares        =('shares', 'mean'),
    avg_saves         =('saves', 'mean'),
    avg_engagement    =('engagement_rate', 'mean'),
    viral_rate        =('is_viral', 'mean'),
).round(2)

# Normalise 0-1 per column for heatmap
niche_norm = (niche_metrics - niche_metrics.min()) / (niche_metrics.max() - niche_metrics.min())

fig, ax = plt.subplots(figsize=(13, 7))
sns.heatmap(niche_norm.T, ax=ax, cmap='RdYlGn',
            annot=niche_metrics.T.round(1), fmt='g',
            linewidths=0.5, linecolor='#0f0f0f',
            annot_kws={'size': 8, 'color': 'white'},
            cbar_kws={'label': 'Normalised score (0-1)'})
ax.set_title('🎯 Niche Performance Heatmap (raw values annotated)', color='white', fontsize=13)
ax.set_ylabel('Metric', color='white')
ax.set_xlabel('Niche', color='white')
plt.xticks(rotation=30, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('tiktok_niche_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop niches by avg engagement rate:')
print(niche_metrics.sort_values('avg_engagement', ascending=False)[['avg_views', 'avg_engagement', 'viral_rate']].to_string())

## 8. Caption Sentiment vs Engagement

In [ ]:
try:
    from transformers import pipeline as hf_pipeline

    sentiment_pipe = hf_pipeline(
        'text-classification',
        model='distilbert-base-uncased-finetuned-sst-2-english',
        truncation=True, max_length=64,
    )

    # Sample 100 captions for speed
    sample = df.sample(100, random_state=42).copy()
    results = sentiment_pipe(list(sample['caption_base']))
    sample['sentiment']       = [r['label'] for r in results]
    sample['sentiment_score'] = [r['score'] if r['label'] == 'POSITIVE'
                                  else -r['score'] for r in results]
    SENTIMENT_AVAILABLE = True
    print(f'Sentiment analysis complete on {len(sample)} captions')
    print(sample['sentiment'].value_counts().to_string())

except Exception as e:
    print(f'Transformers sentiment failed: {e}')
    print('Using rule-based sentiment instead')
    SENTIMENT_AVAILABLE = False

    # Rule-based: positive emojis / words → positive
    pos_words = ['amazing', 'love', 'best', 'great', 'good', '💪', '✨', '🔥', '😍', '❤️', '😂']
    neg_words = ['failing', 'broke', 'sad', 'bad', 'terrible', 'worst', '😭', '💔']

    sample = df.sample(200, random_state=42).copy()

    def rule_sentiment(text):
        text = text.lower()
        pos = sum(w in text for w in pos_words)
        neg = sum(w in text for w in neg_words)
        if pos > neg:   return 'POSITIVE', 0.7 + pos * 0.05
        elif neg > pos: return 'NEGATIVE', -(0.7 + neg * 0.05)
        else:           return 'NEUTRAL',  0.0

    sentiments = sample['caption_base'].apply(rule_sentiment)
    sample['sentiment']       = [s[0] for s in sentiments]
    sample['sentiment_score'] = [s[1] for s in sentiments]

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 6))

# Sentiment distribution
ax = axes[0]
sent_counts = sample['sentiment'].value_counts()
colors_sent = {'POSITIVE': '#25f4ee', 'NEGATIVE': '#fe2c55', 'NEUTRAL': '#ffffff'}
wedge_colors = [colors_sent.get(s, '#888') for s in sent_counts.index]
ax.pie(sent_counts.values, labels=sent_counts.index, autopct='%1.1f%%',
       colors=wedge_colors, textprops={'color': 'white'},
       wedgeprops={'edgecolor': '#0f0f0f', 'linewidth': 2})
ax.set_title('Caption Sentiment Distribution', color='white')

# Sentiment vs ER
ax = axes[1]
for sent, color in colors_sent.items():
    subset = sample[sample['sentiment'] == sent]
    ax.scatter(subset['sentiment_score'], subset['engagement_rate'],
               color=color, alpha=0.6, s=30, label=sent)
ax.set_xlabel('Sentiment Score', color='white')
ax.set_ylabel('Engagement Rate (%)', color='white')
ax.set_title('Sentiment Score vs Engagement Rate', color='white')
ax.legend()

# Mean ER per sentiment
ax = axes[2]
er_by_sent = sample.groupby('sentiment')['engagement_rate'].mean().sort_values(ascending=False)
bar_cols = [colors_sent.get(s, '#888') for s in er_by_sent.index]
bars = ax.bar(er_by_sent.index, er_by_sent.values, color=bar_cols, alpha=0.85)
ax.set_ylabel('Mean Engagement Rate (%)', color='white')
ax.set_title('Engagement Rate by Sentiment', color='white')
for bar, val in zip(bars, er_by_sent.values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.05,
            f'{val:.1f}%', ha='center', color='white', fontsize=10)

plt.suptitle('📝 Caption Sentiment vs TikTok Engagement', color='white', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('tiktok_sentiment_vs_er.png', dpi=150, bbox_inches='tight')
plt.show()

print('Mean ER by sentiment:')
print(er_by_sent.to_string())

## 9. Video Duration vs Performance

In [ ]:
dur_analysis = df.groupby('duration_sec').agg(
    mean_er   =('engagement_rate', 'mean'),
    mean_views=('views', 'mean'),
    mean_saves=('saves', 'mean'),
    viral_rate=('is_viral', 'mean'),
    count     =('post_id', 'count')
).round(3)

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('⏱️ Video Duration vs Performance', color='white', fontsize=14)

for ax, col, title, color in zip(
    axes.flat,
    ['mean_er', 'mean_views', 'mean_saves', 'viral_rate'],
    ['Avg Engagement Rate (%)', 'Avg Views', 'Avg Saves', 'Viral Rate'],
    TIKTOK_COLORS,
):
    bars = ax.bar([str(d)+'s' for d in dur_analysis.index], dur_analysis[col],
                  color=color, alpha=0.85)
    ax.set_title(title, color='white')
    ax.set_xlabel('Duration', color='white')
    for bar, val in zip(bars, dur_analysis[col]):
        ax.text(bar.get_x() + bar.get_width()/2, val * 1.02,
                f'{val:.2f}', ha='center', color='white', fontsize=9)

plt.tight_layout()
plt.savefig('tiktok_duration_performance.png', dpi=150, bbox_inches='tight')
plt.show()
print(dur_analysis.to_string())

## 10. Top Performing Posts & Key Takeaways

In [ ]:
print('='*70)
print('  TOP 10 POSTS BY ENGAGEMENT RATE')
print('='*70)
top10 = (df.nlargest(10, 'engagement_rate')
           [['niche', 'caption_base', 'views', 'engagement_rate',
             'duration_sec', 'n_hashtags', 'is_viral']]
           .reset_index(drop=True))
for i, row in top10.iterrows():
    print(f"\n#{i+1} [{row['niche']}] {row['caption_base'][:60]}...")
    print(f"     Views: {row['views']:,} | ER: {row['engagement_rate']:.1f}% | "
          f"{row['duration_sec']}s | {row['n_hashtags']} hashtags | Viral: {row['is_viral']}")

In [ ]:
# Correlation matrix
num_cols = ['views', 'likes', 'comments', 'shares', 'saves',
            'engagement_rate', 'n_hashtags', 'caption_len', 'duration_sec']
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, ax=ax, mask=mask, cmap='coolwarm', center=0,
            annot=True, fmt='.2f', annot_kws={'size': 9},
            linewidths=0.5, linecolor='#0f0f0f',
            cbar_kws={'label': 'Pearson correlation'})
ax.set_title('Correlation Matrix — TikTok Metrics', color='white', fontsize=13)
plt.tight_layout()
plt.savefig('tiktok_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print('='*70)
print('  KEY FINDINGS — TIKTOK CREATOR ANALYSIS')
print('='*70)

best_niche     = niche_metrics['avg_engagement'].idxmax()
best_niche_er  = niche_metrics['avg_engagement'].max()
best_hour      = df.groupby('post_hour')['engagement_rate'].mean().idxmax()
best_day_val   = df.groupby('post_day', observed=True)['engagement_rate'].mean().idxmax()
best_dur       = dur_analysis['mean_er'].idxmax()
best_hash_bkt  = df.groupby('hashtag_bucket', observed=True)['engagement_rate'].mean().idxmax()

print(f"""
1. BEST NICHE FOR ENGAGEMENT:    {best_niche} (avg ER = {best_niche_er:.1f}%)
2. BEST TIME TO POST:            {best_hour}:00 (highest avg engagement)
3. BEST DAY TO POST:             {best_day_val}
4. BEST VIDEO DURATION:          {best_dur} seconds
5. OPTIMAL HASHTAG COUNT:        {best_hash_bkt}
6. VIRAL RATE IN DATASET:        {df['is_viral'].mean()*100:.1f}% of posts
7. MEDIAN ENGAGEMENT RATE:       {df['engagement_rate'].median():.2f}%
8. MEDIAN VIEWS:                 {int(df['views'].median()):,}
""")

print('Correlation with engagement rate:')
er_corr = corr['engagement_rate'].drop('engagement_rate').sort_values(key=abs, ascending=False)
print(er_corr.to_string())

---
## Summary

This notebook analysed **500 TikTok creator posts** across 10 niches, covering:

- **EDA**: views, likes, comments, shares, saves, engagement rate distributions
- **Hashtag analysis**: frequency counts, word cloud, optimal hashtag count
- **Google Trends**: real-time search interest for creator keywords (pytrends)
- **Caption length**: relationship between word count and engagement
- **Posting time**: best hour and day-of-week heatmap
- **Niche heatmap**: comparative performance across all 10 niches
- **Sentiment analysis**: positive vs negative caption sentiment vs engagement
- **Duration analysis**: 15s vs 60s vs 180s video performance

These insights power the **TikTok Creator PA** tool — run `tiktok_creator_pa.py` to use the AI assistant.

---
*TikTok Creator PA · Radowana Sradowana · University of Hull · 2026*